# Batching: how much faster, how much memory, and are the answers the same?

The queries of one photo (headings and scales) are independent, so the expensive network calls can be batched, like the samples of a batch in training. Three settings, all in `SinglePhotoPipeline` / `HeadingQueryPipeline`:

| setting | what it batches | default |
|---|---|---|
| `loftr_batch` | LoFTR picture pairs per network call | 1 (one pair per call, as before) |
| `dino_batch` | DINO map windows per call | 25 |
| `query_group` | queries whose LoFTR pairs are matched together (heading pipelines) | 1 |

The settings change **speed and memory only**. This notebook checks that on real photos: the time of the stage, the GPU memory peak, and whether the inlier counts are identical to the one-pair-per-call result.

Timing is on this computer's GPU, which other programs may share, so repeat a number before trusting a small difference.

# Part 1 · Settings

In [1]:
import os, sys, math, time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

# ------------------------------------------------------------------ settings: change here, run again
N_PHOTOS = 60               # photos loaded (only the first few are used)
N_TEST = 3                  # photos used for the timings
HEADINGS = [-5.0, 0.0, 5.0] # the queries of one photo: 3 headings x 2 scales = 6 queries
SCALE_FACTORS = [1.0, 1.05]
K_PEAKS = 50
M_FINE = 10
LOFTR_BATCHES = [1, 2, 5, 10, 20, 30, 60]   # LoFTR pairs per call tried (60 = all pairs of one photo's 6 queries)
DINO_BATCHES = [1, 5, 25, 50]               # DINO windows per call tried
QUERY_GROUPS = [1, 3, 6]                    # queries matched together tried (with the largest LoFTR batch that fits)
REPEATS = 2                 # every timing is the best of this many runs
SEED = 958476

# Part 2 · Set up and prepare the queries once

In [2]:
import torch
from single_photo_pipeline import STAGES
from heading_query_pipeline import HeadingQueryPipeline
from header.settings import DEVICE
from image_processing.map_levels import level_for_height
from coarse_search.heading_queries import query_geometry
from fine_matching.loftr_matcher import candidate_matches_group, stage2_from_matches
from semantic_rerank.dino_rerank import dino_candidate_scores

pipeline = HeadingQueryPipeline("official", "edge_zncc", headings=HEADINGS, scale_factors=SCALE_FACTORS, k_peaks=K_PEAKS, m_fine=M_FINE,
                                matcher="loftr_minima", matcher_confidence=0.21, loftr_coarse_threshold=0.21, min_inliers=8, satellite="new")
pipeline.setup(N_PHOTOS, SEED)
for line in pipeline.describe():
    print(line)

def clock_now():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter()

prepared = {}       # photo -> (level, [prepared query])
for index in range(N_TEST):
    item = pipeline.test_set.items[index]
    level = level_for_height(item["rel_height"], pipeline.camera)
    queries = []
    for query in pipeline.queries:
        queries.append(pipeline.prepare_query(item, pipeline.test_set.priors[index], level, query_geometry(query["heading"], query["scale_factor"])))
    prepared[index] = (level, queries)
pairs_per_photo = sum(int((q["peaks"][:, 1] < 1e8).sum()) for q in prepared[0][1])
print("%d photos x %d queries prepared; %d LoFTR pairs per photo" % (N_TEST, len(pipeline.queries), pairs_per_photo))

level 3: 0.85 m/pixel, picture 16321 x 15893, map coverage 100%
level 4: 1.20 m/pixel, picture 11540 x 11238, map coverage 100%
level 5: 1.70 m/pixel, picture 8160 x 7946, map coverage 100%
level 6: 2.40 m/pixel, picture 5770 x 5619, map coverage 100%
level 7: 3.39 m/pixel, picture 4080 x 3973, map coverage 100%


map store ready, took 119 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data: official | 60 photos available | zoom levels [3, 4, 5, 6, 7]
photos: official | map: new
coarse method: edge_zncc | maps: ['edge_zncc']
semantic layer: on (best K=50 -> M=10) | DINO in the decision: on (gamma 1.00, rescue (None, None))
fine matcher: loftr_minima | answer bar: 8 inliers | RANSAC 3.0 px | matcher confidence 0.21 | LoFTR coarse threshold 0.21, upscale 1.0 | border 0 px, margin 0 px
DINO order weight: 100.0 | projection overrides: none (settings.GEOMETRY)
batches: 1 LoFTR pairs and 25 DINO places per call
device: cuda
queries per photo: 6 (headings [-5.0, 0.0, 5.0], scale factors [1.0, 1.05]); K and M are per query; 1 queries matched together


3 photos x 6 queries prepared; 60 LoFTR pairs per photo


# Part 3 · LoFTR pairs per call

All the pairs of one photo (every candidate of its 6 queries) are matched with each `loftr_batch`. `seconds per pair` is the time of the network and the pair handling, before RANSAC. The **identical** column compares the inlier counts of every candidate with the one-pair-per-call run.

In [3]:
def fine_stage_for_photo(index, batch_size):
    level, queries = prepared[index]
    jobs = [dict(canvas=q["canvas"], valid=q["valid"], level=level, peaks=q["peaks"]) for q in queries]
    pipeline.matcher.batch_size = batch_size
    stored = candidate_matches_group(pipeline.matcher, jobs, pipeline.level_images)
    inliers = []
    for kept in stored:
        inliers.extend(stage2_from_matches(kept, level, pipeline.matcher.min_confidence)["inliers"].tolist())
    return np.array(inliers)

reference = {index: fine_stage_for_photo(index, 1) for index in range(N_TEST)}
rows = []
for batch_size in LOFTR_BATCHES:
    best = None
    peak_mb = np.nan
    different = 0
    status = "ok"
    try:
        for _ in range(REPEATS):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            clock = clock_now()
            counts = [fine_stage_for_photo(index, batch_size) for index in range(N_TEST)]
            seconds = (clock_now() - clock) / N_TEST
            peak_mb = torch.cuda.max_memory_allocated() / 1e6
            if best is None or seconds < best:
                best = seconds
        for index in range(N_TEST):
            different += int((counts[index] != reference[index]).sum())
    except torch.cuda.OutOfMemoryError:
        status = "out of memory"
        torch.cuda.empty_cache()
    rows.append({"loftr_batch": batch_size, "status": status, "seconds per photo": best, "seconds per pair": None if best is None else best / pairs_per_photo,
                 "GPU peak (MB)": peak_mb, "candidates with another inlier count": different})
loftr_table = pd.DataFrame(rows)
base_seconds = loftr_table["seconds per photo"].iloc[0]
loftr_table["speedup"] = base_seconds / loftr_table["seconds per photo"]
display(loftr_table.round(4))

,loftr_batch,status,seconds per photo,seconds per pair,GPU peak (MB),candidates with another inlier count,speedup
0,1,ok,2.1570,0.0360,323.2599,0,1.0000
1,2,ok,2.0324,0.0339,480.4111,0,1.0613
2,5,ok,1.9025,0.0317,951.8648,0,1.1338
3,10,ok,1.8430,0.0307,1737.6210,0,1.1704
4,20,ok,1.8336,0.0306,3309.1333,0,1.1764
5,30,out of memory,NaN,NaN,NaN,0,NaN
6,60,out of memory,NaN,NaN,NaN,0,NaN


# Part 4 · DINO windows per call

The DINO scores of the K = 50 shortlist places of one query, with each `dino_batch`. The scores should be the same up to the last decimals.

In [4]:
level0, queries0 = prepared[0]
short = queries0[0]["shortlist"]
canvas, valid = queries0[0]["canvas"], queries0[0]["valid"]
reference_dino = dino_candidate_scores(pipeline.encoder, canvas, valid, pipeline.level_images, level0, short, 25)[1]
rows = []
for batch_size in DINO_BATCHES:
    best = None
    peak_mb = np.nan
    for _ in range(REPEATS + 1):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        clock = clock_now()
        dense = dino_candidate_scores(pipeline.encoder, canvas, valid, pipeline.level_images, level0, short, batch_size)[1]
        seconds = clock_now() - clock
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        if best is None or seconds < best:
            best = seconds
    rows.append({"dino_batch": batch_size, "seconds per query": best, "GPU peak (MB)": peak_mb, "largest score difference": float(np.nanmax(np.abs(dense - reference_dino)))})
dino_table = pd.DataFrame(rows)
display(dino_table.round(5))

,dino_batch,seconds per query,GPU peak (MB),largest score difference
0,1,0.13700,169.90976,0.00007
1,5,0.10011,183.28730,0.00007
2,25,0.08491,252.45850,0.00000
3,50,0.08149,299.47187,0.00000


# Part 5 · A whole photo: `loftr_batch` and `query_group` together

`run_queries` for one photo (6 queries): render, maps, coarse, DINO per query, and the LoFTR pairs of `query_group` queries together. The answer should not change: the check compares the best inliers of every query with the plain setting.

In [5]:
def run_photo(index, loftr_batch, query_group):
    pipeline.matcher.batch_size = loftr_batch
    pipeline.query_group = query_group
    item = pipeline.test_set.items[index]
    clock = clock_now()
    found = pipeline.run_queries(item, pipeline.test_set.priors[index])
    seconds = clock_now() - clock
    inliers = np.concatenate([np.asarray(q["stage2"]["inliers"]) for q in found["queries"]])
    return seconds, inliers

run_photo(0, 1, 1)      # warm-up
reference_full = {index: run_photo(index, 1, 1)[1] for index in range(N_TEST)}
rows = []
for query_group in QUERY_GROUPS:
    for loftr_batch in sorted(set([1, M_FINE, M_FINE * query_group])):
        status = "ok"
        best = None
        peak_mb = np.nan
        different = 0
        try:
            torch.cuda.reset_peak_memory_stats()
            times = []
            for index in range(N_TEST):
                seconds, inliers = run_photo(index, loftr_batch, query_group)
                times.append(seconds)
                different += int((inliers != reference_full[index]).sum())
            best = float(np.mean(times))
            peak_mb = torch.cuda.max_memory_allocated() / 1e6
        except torch.cuda.OutOfMemoryError:
            status = "out of memory"
            torch.cuda.empty_cache()
        rows.append({"query_group": query_group, "loftr_batch": loftr_batch, "status": status, "seconds per photo": best, "GPU peak (MB)": peak_mb,
                     "candidates with another inlier count": different})
full_table = pd.DataFrame(rows)
full_table["speedup"] = full_table["seconds per photo"].iloc[0] / full_table["seconds per photo"]
display(full_table.round(3))

,query_group,loftr_batch,status,seconds per photo,GPU peak (MB),candidates with another inlier count,speedup
0,1,1,ok,4.424,2921.393,0,1.000
1,1,10,ok,4.078,2921.393,0,1.085
2,3,1,ok,4.428,2921.393,0,0.999
3,3,10,ok,4.099,2921.393,0,1.079
4,3,30,out of memory,NaN,NaN,0,NaN
5,6,1,ok,4.433,2921.393,0,0.998
6,6,10,ok,4.092,2921.393,0,1.081
7,6,60,out of memory,NaN,NaN,0,NaN


# Part 6 · Save

In [6]:
out_dir = ROOT / "tests" / "Batch_Speed_Check"
out_dir.mkdir(parents=True, exist_ok=True)
loftr_table.to_csv(out_dir / "loftr_batch.csv", index=False)
dino_table.to_csv(out_dir / "dino_batch.csv", index=False)
full_table.to_csv(out_dir / "whole_photo.csv", index=False)
print("saved to", out_dir)

saved to /home/voidness/Storage/Synced Folders/InterIIT Tech 15/IdeaForge-GeoLoc-InterIIT15/tests/Batch_Speed_Check


**How to read it.** Pick the largest `loftr_batch` and `query_group` that keep the GPU peak well under the memory the program can use (the other programs on the GPU count too; on the Jetson the memory is shared with the CPU) and that show no candidate with another inlier count. A speedup that flattens between two batch sizes means the GPU is already full: take the smaller one.